<a href="https://colab.research.google.com/github/lspnzz/granted-search-engine/blob/main/evals/notebooks/run_search_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Granted Search Evals**

The most important thing for us is that our service retrieves all possible grants that could be relevant for each pitch. We're ok with getting less relevant grants, as long as we don't miss relevant ones.

We only care about document-level retrieval, we don't care about specific sections of the document.

First we need to make sure our app does what it's supposed to do, then we'll figure out how to make it more precise and efficient.

### **Primary metric**
- **Recall:** measures how many of the relevant documents were successfully retrieved. It focuses on not missing important results. Higher recall means fewer relevant documents were left out.

### **Secondary metrics**

We will evaluate these metrics at a later time, once we've improved the effectiveness of the system.

- **context relevance:** make sure all the retrieved chunks make sense for the given input;
- **context precision:** more relevant chunks are ranked higher than others;


## **Run configuration**

In [1]:
from datetime import date
today = date.today()
date_str = today.strftime("%Y-%m-%d")

In [2]:
MODEL_NAME = input("Enter the model name: ")
MODEL_DIMENSIONS = input("Enter the model dimensions: ")
METRIC = input("Enter the distance metric: ")

Enter the model name: text-embedding-3-small
Enter the model dimensions: 1536
Enter the distance metric: cosine


In [3]:
CHUNK_SIZE = input("Enter the chunk size: ")
CHUNK_OVERLAP = input("Enter the chunk overlap: ")

Enter the chunk size: 2000
Enter the chunk overlap: 200


In [4]:
k = int(input("Enter the number of results to retrieve: "))

Enter the number of results to retrieve: 23


In [5]:
RUN_ID = f"{date_str}_model-{MODEL_NAME}_dimensions-{MODEL_DIMENSIONS}_metric-{METRIC}_chunk-size-{CHUNK_SIZE}_chunk-overlap-{CHUNK_OVERLAP}_top-{k}"

### **Run eval pipeline**

In [6]:
from google.colab import userdata
import json

PINECONE_INDEX_NAME = f"eval-model-{MODEL_NAME}-{MODEL_DIMENSIONS}"
PINECONE_NAMESPACE = f"eval-chunk-size{CHUNK_SIZE}-overlap-{CHUNK_OVERLAP}"

params = {
    "load_grants_from_file": "filtered_raw_grants.json",  # (LS): Curated set of grants for the Golden set
    "pinecone_index_name": PINECONE_INDEX_NAME,
    "pinecone_namespace": PINECONE_NAMESPACE,
    "model_name": MODEL_NAME,
    "dimensions": MODEL_DIMENSIONS,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
}

json_data = json.dumps(params)
url = userdata.get("dataPipelineFunctionUrl")

!curl -X POST "{url}" \
  -H "Content-Type: application/json" \
  -d '{json_data}'

Pipeline completed successfully

## **Run searches for the Golden Dataset**

In [7]:
def search_grants(pitch_text, k):
    search_url = userdata.get("searchFunctionUrl")
    HEADERS = {"Content-Type": "application/json"}
    payload = {
        "pitch": pitch_text,
        "top_k": k,
        "model_name": MODEL_NAME,
        "dimensions": MODEL_DIMENSIONS,
        "pinecone_index_name": PINECONE_INDEX_NAME,
        "pinecone_namespace": PINECONE_NAMESPACE
    }

    response = requests.post(search_url, headers=HEADERS, json=payload)
    data = response.json()
    return data.get("grants", [])


In [8]:
from tqdm import tqdm
import pandas as pd
import requests

# (LS): Load Golden dataset from Google Drive.
file_id = "1B0QZQf6xDjj01ViGjYY33J_D_TVB-LOf"  # the part after /d/ and before /view
golden_url = f"https://drive.google.com/uc?export=download&id={file_id}"
golden_df = pd.read_csv(golden_url, sep=";")

train_size = 0.7
train_df = golden_df.sample(frac=train_size, random_state=42)
results = []

for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):
    pitch_id = row["pitch_id"]
    pitch_text = row["pitch"]

    try:
        matched_grants = search_grants(pitch_text, k)
        results.append({
            "pitch_id": pitch_id,
            "matched_grant_chunks_ids": [grant["id"] for grant in matched_grants],
        })

    except Exception as e:
        print(f"Error processing pitch {pitch_id}: {e}")

run_df = pd.DataFrame(results)
run_df.to_csv(f"{RUN_ID}.csv")

100%|██████████| 53/53 [01:45<00:00,  1.98s/it]


## **Compute Recall@k**

Process retrieved grant chunks to extract the grant ID:

In [9]:
def get_grant_id_from_chunk(chunk_id):
    if isinstance(chunk_id, str) and '-' in chunk_id:
        return chunk_id.rsplit('-', 1)[0]
    return chunk_id

run_df["matched_grant_ids"] = run_df["matched_grant_chunks_ids"].apply(lambda x: [get_grant_id_from_chunk(chunk_id) for chunk_id in x])

Merge the run results with the expected answers from the Golden dataset:

In [10]:
merged_df = pd.merge(train_df, run_df, on="pitch_id", how="inner")

We'll temporarily exclude cases where no relevant chunks should have been returned. We'll evaluate these cases separately once the system is updated to handle them ("False alarm rate" on the negative set).

> **TODO(LS):** Update search engine to handle "no relevant grants" case.

In [11]:
merged_df = merged_df[merged_df["matching_grant_ids"].notna()]

Compute the recall:

In [12]:
def compute_recall(row):
    true_grants = set(row["matching_grant_ids"].split(", "))
    retrieved_grants = set(row["matched_grant_ids"])

    intersection = len(true_grants.intersection(retrieved_grants))
    return intersection / len(true_grants)

merged_df["recall"] = merged_df.apply(compute_recall, axis=1)
print(f"Mean Recall: {merged_df['recall'].mean():.4f}")

Mean Recall: 0.4742


Save results (remember to safely store downloaded file):



In [13]:
from google.colab import files

results_filename = f"{RUN_ID}_recall.csv"
results_df = merged_df[['pitch_id', 'recall']]
results_df.to_csv(results_filename, index=False)
files.download(results_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **Pinecone cleanup**

In [43]:
!pip install -q pinecone==8.0.0

In [44]:
from pinecone import Pinecone

PINECONE_API_KEY = userdata.get("pineconeApiKey")
pc = Pinecone(api_key=PINECONE_API_KEY)

if pc.has_index(name=PINECONE_INDEX_NAME):
    pc.delete_index(PINECONE_INDEX_NAME)

ImportError: cannot import name 'Pinecone' from 'pinecone' (unknown location)